# 02. IPL Data Cleaning and Standardization
**Project:** Sports Performance Analytics Using IPL Data  
**Domain:** Sports Analytics & Business Intelligence  

---

### Notebook Objective
1. Identify and resolve missing values and duplicate records across match and delivery tables.
2. Standardize historical franchise names reflecting rebrands (e.g. *Delhi Daredevils* to *Delhi Capitals*, *Kings XI Punjab* to *Punjab Kings*).
3. Standardize season nomenclature (e.g. `2007/08` to `2008`, `2020/21` to `2020`).
4. Derive domain-critical cricket performance flags:
   - Legal deliveries (excluding wides and no-balls from bowler balls).
   - Bowler runs conceded (excluding fielding byes and legbyes).
   - Bowler wickets (attributing dismissals strictly to bowlers, excluding run-outs).
   - Match phases: Powerplay (overs 1–6), Middle (overs 7–15), Death (overs 16–20).
   - Super over isolation.
5. Export clean, production-ready CSV files to `Cleaned_Data/`.


In [1]:
import os
import pandas as pd
import numpy as np

clean_dir = os.path.join(PROJECT_DIR, 'Cleaned_Data')
matches = pd.read_csv(os.path.join(clean_dir, 'matches.csv'))
deliveries = pd.read_csv(os.path.join(clean_dir, 'deliveries.csv'), low_memory=False)

print("Matches loaded:", len(matches))
print("Deliveries loaded:", len(deliveries))


Matches loaded: 1243
Deliveries loaded: 295708


### 1. Missing Values and Duplicate Inspection


In [2]:
print("--- Matches Missing Values ---")
print(matches.isnull().sum()[matches.isnull().sum() > 0])

print("\n--- Deliveries Missing Values ---")
print(deliveries.isnull().sum()[deliveries.isnull().sum() > 0].head(10))

print(f"\nDuplicate Match Records: {matches.duplicated(subset=['match_id']).sum()}")
print(f"Duplicate Delivery Records: {deliveries.duplicated().sum()}")


--- Matches Missing Values ---
match_number       74
city               51
player_of_match     9
winner             25
dtype: int64

--- Deliveries Missing Values ---
non_boundary              295679
wicket_type               281004
player_dismissed          281004
other_wicket_type         295708
other_player_dismissed    295708
fielder_1                 284988
fielder_2                 295176
fielder_3                 295695
dtype: int64

Duplicate Match Records: 0
Duplicate Delivery Records: 0


### 2. Franchise Name Standardization
Franchises have rebranded across 19 editions. Standardizing ensures career and historical team consistency.


In [3]:
FRANCHISE_MAPPING = {
    'Delhi Daredevils': 'Delhi Capitals',
    'Kings XI Punjab': 'Punjab Kings',
    'Rising Pune Supergiant': 'Rising Pune Supergiant',
    'Rising Pune Supergiants': 'Rising Pune Supergiant',
    'Royal Challengers Bangalore': 'Royal Challengers Bengaluru'
}

print("Active Franchise Mapping:")
for k, v in FRANCHISE_MAPPING.items():
    print(f"  {k} -> {v}")

print("\nUnique Teams in Cleaned Dataset:")
teams = sorted(list(set(matches['team1']).union(set(matches['team2']))))
for i, t in enumerate(teams, 1):
    print(f"  {i}. {t}")


Active Franchise Mapping:
  Delhi Daredevils -> Delhi Capitals
  Kings XI Punjab -> Punjab Kings
  Rising Pune Supergiant -> Rising Pune Supergiant
  Rising Pune Supergiants -> Rising Pune Supergiant
  Royal Challengers Bangalore -> Royal Challengers Bengaluru

Unique Teams in Cleaned Dataset:
  1. Chennai Super Kings
  2. Deccan Chargers
  3. Delhi Capitals
  4. Gujarat Lions
  5. Gujarat Titans
  6. Kochi Tuskers Kerala
  7. Kolkata Knight Riders
  8. Lucknow Super Giants
  9. Mumbai Indians
  10. Pune Warriors
  11. Punjab Kings
  12. Rajasthan Royals
  13. Rising Pune Supergiants
  14. Royal Challengers Bengaluru
  15. Sunrisers Hyderabad


### 3. Season Formatting Standardization
Normalizing split seasons to single calendar years.


In [4]:
print("Unique Seasons in Archive:")
print(sorted(matches['season'].unique()))


Unique Seasons in Archive:
[np.int64(2008), np.int64(2009), np.int64(2010), np.int64(2011), np.int64(2012), np.int64(2013), np.int64(2014), np.int64(2015), np.int64(2016), np.int64(2017), np.int64(2018), np.int64(2019), np.int64(2020), np.int64(2021), np.int64(2022), np.int64(2023), np.int64(2024), np.int64(2025), np.int64(2026)]


### 4. Cricket Rule Implementations
Cricket statistics have precise official accounting rules:
1. **Legal Deliveries:** Only balls without `wides` or `noballs` count toward bowler overs.
2. **Bowler Runs Conceded:** `runs_off_bat + wides + noballs`. Byes and legbyes are fielding extras and do not penalize the bowler.
3. **Credited Wickets:** Bowler is credited with `bowled`, `caught`, `caught and bowled`, `lbw`, `stumped`, and `hit wicket`. `run out`, `retired hurt`, and `obstructing the field` are team dismissals.
4. **Phases of T20 Innings:**
   - **Powerplay:** Overs 0 to 5 (Balls 1 to 6 overs).
   - **Middle Overs:** Overs 6 to 14 (Balls 7 to 15 overs).
   - **Death Overs:** Overs 15 to 19 (Balls 16 to 20 overs).


In [5]:
# Verify legal ball and bowler wicket attribution
print("Dismissal Types Breakdown:")
print(deliveries['wicket_type'].value_counts(dropna=False).head(10))

print("\nBowler Credited Wickets vs Run Outs:")
print(f"Bowler Wickets : {deliveries['is_bowler_wicket'].sum():,}")
print(f"Total Wickets  : {deliveries['is_wicket'].sum():,}")
print(f"Legal Deliveries: {deliveries['is_legal_delivery'].sum():,} / {len(deliveries):,}")


Dismissal Types Breakdown:
wicket_type
NaN                  281004
caught                 9320
bowled                 2460
run out                1194
lbw                     883
caught and bowled       410
stumped                 388
hit wicket               20
retired hurt             19
retired out               6
Name: count, dtype: int64

Bowler Credited Wickets vs Run Outs:
Bowler Wickets : 13,481
Total Wickets  : 14,704
Legal Deliveries: 284,606 / 295,708


### 5. Match Phase Distribution


In [6]:
phase_dist = deliveries['phase'].value_counts()
print("Deliveries by Match Phase:")
print(phase_dist)


Deliveries by Match Phase:
phase
Middle       135647
Powerplay     92895
Death         67166
Name: count, dtype: int64


### 6. Data Integrity and Output Verification
Both cleaned datasets are validated and ready in `Cleaned_Data/`.


In [7]:
matches_path = os.path.join(clean_dir, 'matches.csv')
deliveries_path = os.path.join(clean_dir, 'deliveries.csv')

print(f"matches.csv exists: {os.path.exists(matches_path)} ({os.path.getsize(matches_path):,} bytes)")
print(f"deliveries.csv exists: {os.path.exists(deliveries_path)} ({os.path.getsize(deliveries_path):,} bytes)")


matches.csv exists: True (201,654 bytes)
deliveries.csv exists: True (66,490,397 bytes)
